Instructions Answer the following questions using F1 data on the AWS S3 utilizing Databricks. You can use Pandas, R or PySpark. 

[10 pts] What was the average time each driver spent at the pit stop for each race? \
[20 pts] Rank the average time spent at the pit stop in order of who won each race \
[20 pts] Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset \
[20 pts] Who is the youngest and oldest driver for each race? Create a new column called “Age” \
[20 pts] For a given race, which driver has the most wins and losses? \
[10 pts] Continue exploring the data by answering your own question. \
Commit your assignment to your individual Github classroom repo. Your code and git commits should follow the basic principles we discussed so far.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.functions import avg, current_date, col, year, date_diff,floor, count
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
df_pitstops= spark.read.csv('s3://columbia-gr5069-main/raw/pit_stops.csv', 
                            header=True) #reading in file from the db on aws
display(df_pitstops)                   

#Q1: Average Time each driver spent at the pitstop for each race

In [0]:
#Converting to float as it isn't in float type
df_pitstops = df_pitstops.withColumn('milliseconds', df_pitstops['milliseconds'].cast(FloatType()))
#Calc avg pit stop time by race and driver Ids:
pitstops_driver_avg_time = df_pitstops.groupBy('raceId','driverId').agg((avg('milliseconds')).cast(FloatType()).alias('avg_pit_milliseconds')) #avg column name as .alias('avg_milliseconds')
display(pitstops_driver_avg_time)

#Q2 Rank the average time spent at the pit stop in order of who won each race
rank the average time spent at the pit stop (from previous question) and use the finishing order on the race to rank order these average times. Add in your comments how you decided to deal with drivers who did not finish the race

In [0]:
race_results = spark.read.csv('s3://columbia-gr5069-main/raw/results.csv', header=True)
display(race_results)

To determine winner: Look at position with raceID. position =1 is winner.

In [0]:
# Get the latest lap for each race and driver (as position in latest lap indicates win position as indicated in slack message)
latest_lap = race_results.groupBy('raceId', 'driverId').agg(F.max('laps').alias('latest_lap'))

# Join the latest_lap information with the original race_results DataFrame
latest_lap_join = race_results.join(latest_lap, ['raceId', 'driverId'])

# Filter the rows where laps equals latest_lap
latest_lap_result = latest_lap_join.filter(latest_lap_join.laps == latest_lap_join.latest_lap)

# Select all columns from the filtered result
display(latest_lap_result)

In [0]:
#using the latest_lap df to join with pitstops_driver_avg_time--- we get the position of driver in the latest lap of each race (which determines who won-- accoriding to slack message) combined with avg pitstop time. 

result_driver = latest_lap_result.withColumn('win_position', latest_lap_result.position).select('raceId','driverId','win_position').distinct()
pitstop_results_df = pitstops_driver_avg_time.join(result_driver, on=['raceId','driverId'], how='inner').orderBy('win_position')
display(pitstop_results_df)

I am going to assign the rank 'DNF' to drivers who didn't finish the race. 'win_position' = null/none for DNF

In [0]:
#overall ranking by pit stop times
window_spec = Window.orderBy('avg_pit_milliseconds')  #ordering  by time in pitstop
# ranking across ALL drivers for their pit stop times. 
ranked_pitstop_results_df = (pitstop_results_df
    .withColumn('overall_pit_rank', F.row_number().over(window_spec))
    .orderBy('win_position')  # Final sort by race finishing order
)

# Handle DNF drivers (assuming 'win_position' = null/none for DNF)
ranked_pitstop_results_df = ranked_pitstop_results_df.withColumn(
    'race_position',
    F.when(F.col('win_position').isNull(), 'DNF')
     .otherwise(F.col('win_position'))
)

# Display the result
display(ranked_pitstop_results_df)

#Q3 Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset
insert a three letter code where codes are missing in the driver dataset. Write in comments how you arrived at these codes

In [0]:
df_drivers= spark.read.csv('s3://columbia-gr5069-main/raw/drivers.csv', 
                            header=True)
display(df_drivers)

I'm just taking the first three letters of a driver's surname and inserting it as the 'fixed_code' for all those who have na/null Codes.

In [0]:
#taking first three letters of last name and capitalising it for missing code
#df_drivers = df_drivers.withColumn('code', F.concat(F.substring(F.upper(F.col('surname')), 0, 3), F.lit(' ')))

df_drivers = df_drivers.withColumn(
    'fixed_code',
    F.when(F.col('code')== r'\N',F.concat(F.substring(F.upper(F.col('surname')), 0, 3)))
           .otherwise(F.col('code'))
    )
    
display(df_drivers)

#Q4 Who is the youngest and oldest driver for each race? Create a new column called “Age”
- create a new column called “Age” that counts how many birthdays each driver has had in their lives
- explain how you reached this number in your comments
- identify the oldest and youngest driver for each race

Joining race_results and df_drivers (we need joined data for future Qs)

In [0]:
#attaching the name and age column to the ranked_pitstop_results_df
race_results = race_results.join(df_drivers, on='driverId', how='inner')
display(race_results)

I'm calculating the age by taking the floor of the DIFFERENCE between the current date and the dob of each driver-- then dividing by 365 to get the age in years. 

In [0]:
#calclating AGE for each driver and adding it to the race_results df
race_results = race_results.withColumn('age', floor(date_diff(current_date(), 'dob') / 365))
display(race_results)

I'm calcualting oldest and youngest driver: grouping by raceID, then getting the max of the column age, and getting the name of the driver with the max age--- similarly for min age. 

In [0]:
#For each race ID, creating a df with oldest driver and youngest drivers' name and age
oldest_youngest_df = race_results.groupBy('raceId').agg(F.max(F.col('age')).alias('oldest_age'), F.max(F.col('surname')).alias('oldest_surname'), F.min(F.col('age')).alias('youngest_age'), F.min(F.col('surname')).alias('youngest_surname'))
display(oldest_youngest_df)

#Q5 For a given race, which driver has the most wins and losses?
on a given race, provide a count of how many times a diver has won previous races, and a count of all the times a driver has not won (but completed) a race

In [0]:
#calculating which driver has most wins or loses for a race_id from results_df
# Define window for driver's career up to CURRENT race
window_spec = (Window
    .partitionBy('driverId')
    .orderBy('raceId')
    .rowsBetween(Window.unboundedPreceding, -1)  # Exclude current race-- counting wins for all races preceeding current race! (for cumulative count)
)
#creating column total_wins for all wins before current race (basically, this is the cumulative win count). same for losses.
wins_losses_df = pitstop_results_df.withColumn(
    'total_wins', 
    F.sum(F.when(F.col('win_position') == 1, 1).otherwise(0)).over(window_spec) #adding 1 to total_wins everytime win_positon=1
).withColumn(
    'career_losses',
    F.sum(
        F.when(
            (F.col('win_position') > 1) &  # Finished race but loss!
            (F.col('win_position').isNotNull()),  # When race wasn't finished, we don't want it in loss count!-- Exclude DNFs
            1
        ).otherwise(0)
    ).over(window_spec)
)
display(wins_losses_df)


#6 Continue exploring the data by answering your own question.:which lap was the fastest lap for winners?

In [0]:
#figuring out which lap was fastest_lap for winners from race_results
fastest_lap_df = race_results.select('raceId','driverId','position','fastestLapTime', 'fastestLap').where(F.col('position') == 1)
display(fastest_lap_df)

In [0]:
#How many fastest laps are after the first 10 laps?
fastest_lap_df.where(F.col('fastestLap') > 10).count() 

In [0]:
#How many earlier in the race?
fastest_lap_df.where(F.col('fastestLap') <= 10).count() 

Looks like most of the fastest laps for winners are later in the race! 